# 8.18 — POS tagging and morphology

POS tagging names a word's syntactic job, while morphology records the word-internal clues that often reveal that job.

Suffix features, emission scores, and contextual tag transitions combine to tag words as nouns, verbs, adjectives, and richer morphology bundles before parsing.

Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 713
random.seed(SEED)
np.random.seed(SEED)


def softmax(values):
    arr = np.array(values, dtype=float)
    shifted = arr - np.max(arr)
    exp = np.exp(shifted)
    return exp / exp.sum()


def accuracy(pred, gold):
    return sum(int(a == b) for a, b in zip(pred, gold)) / max(1, len(gold))


def show_table(rows, headers):
    widths = [len(str(h)) for h in headers]
    for row in rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))
    print(" | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers)))
    print("-+-".join("-" * w for w in widths))
    for row in rows:
        print(" | ".join(str(value).ljust(widths[i]) for i, value in enumerate(row)))


def make_sequence_ladder(topic):
    if topic == "8.13":
        return [
            {"name": "D1 reversal", "sequences": [["dog", "bites"], ["bites", "dog"]], "labels": [1, 0], "max_pos": 2},
            {"name": "D2 clean order", "sequences": [["a", "then", "b"], ["b", "then", "a"], ["red", "before", "blue"], ["blue", "before", "red"], ["left", "right"]], "labels": [1, 0, 1, 0, 1], "max_pos": 5},
            {"name": "D3 agreement", "sequences": [["dogs", "chase", "cat"], ["cat", "chase", "dogs"], ["he", "likes", "tea"], ["tea", "likes", "he"], ["birds", "fly", "today"]], "labels": [1, 0, 1, 0, 1], "max_pos": 8},
            {"name": "D4 pair tasks", "sequences": [["subject", "verb", "object", "now"], ["object", "verb", "subject", "now"], ["query", "key", "value", "end"], ["value", "key", "query", "end"], ["first", "middle", "last", "stop"]], "labels": [1, 0, 1, 0, 1], "max_pos": 10},
            {"name": "D5 beyond learned", "sequences": [["start"] + ["filler"] * n + ["end"] for n in [4, 8, 12, 16, 20]], "labels": [1, 1, 1, 1, 1], "max_pos": 22},
        ]
    if topic == "8.14":
        return [
            {"name": "D1 mask toy", "T": 32, "w": 2, "evidence": [(19, 0)], "global_tokens": []},
            {"name": "D2 local deps", "T": 24, "w": 2, "evidence": [(4, 3), (8, 6), (12, 11), (16, 15), (20, 18)], "global_tokens": []},
            {"name": "D3 global random", "T": 40, "w": 2, "evidence": [(30, 0), (35, 1), (21, 18), (10, 8), (39, 3)], "global_tokens": [0, 1]},
            {"name": "D4 document snippets", "T": 64, "w": 3, "evidence": [(50, 2), (60, 4), (32, 31), (12, 10), (45, 5)], "global_tokens": [0, 2, 4]},
            {"name": "D5 far evidence", "T": 96, "w": 3, "evidence": [(90, 0), (88, 6), (70, 9), (44, 43), (30, 27)], "global_tokens": [0, 6, 9]},
        ]
    if topic == "8.15":
        return [
            {"name": "D1 logits", "rows": [[3.0, 2.0, 0.5]], "target": ["A"]},
            {"name": "D2 clean rows", "rows": [[3, 1, 0], [0, 3, 1], [1, 0, 3], [4, 2, 0], [0, 4, 2]], "target": list("ABCAB")},
            {"name": "D3 traps", "rows": [[2.1, 2.0, 0], [1.8, 1.7, 1.6], [3, 2.9, 0], [2, 2, 1.9], [0, 2.2, 2.1]], "target": list("BACCB")},
            {"name": "D4 candidates", "rows": [[2.5, 2.4, 0.1], [0.5, 2.0, 1.9], [2.2, 2.1, 2.0], [3.0, 0.2, 0.1], [0.1, 2.8, 2.7]], "target": list("ABABB")},
            {"name": "D5 long generation", "rows": [[2.0, 1.9, 0.2], [1.5, 1.4, 2.2], [2.1, 2.0, 0.4], [0.4, 2.2, 2.1], [1.9, 1.8, 0.7], [0.5, 2.4, 2.3], [2.3, 2.2, 0.2]], "target": list("ACABABB")},
        ]
    if topic == "8.16":
        return [
            {"name": "D1 lesson pair", "cand": "the cat sat", "ref": "the cat slept", "probs": [0.5, 0.25, 0.25]},
            {"name": "D2 paraphrases", "pairs": [("small cat sleeps", "small cat sleeps"), ("red bird flies", "red bird sings"), ("quick dog runs", "dog runs quickly"), ("bright moon rises", "moon rises bright"), ("blue car stops", "blue bus stops")]},
            {"name": "D3 predicate order", "pairs": [("the cat chased mouse", "the mouse chased cat"), ("doctor treats patient", "doctor helps patient"), ("team wins final", "team loses final"), ("user resets password", "password resets user"), ("model answers question", "model answers query")]},
            {"name": "D4 summaries", "pairs": [("sales rose in june", "sales rose in june after ads"), ("server failed at noon", "server recovered at noon"), ("policy changed for users", "policy changed for all users"), ("campaign reached creators", "campaign reached many creators"), ("video views increased", "video watch time increased")]},
            {"name": "D5 long refs", "pairs": [("the model answered safely with citations", "the model answered correctly with citations"), ("the summary covered three facts and omitted risk", "the summary covered three facts and named risk"), ("translation kept names but changed tense", "translation kept names and preserved tense"), ("caption mentions dog running near beach", "caption mentions dog walking near beach"), ("assistant refused unsafe request politely", "assistant refused unsafe request clearly")]},
        ]
    if topic == "8.17":
        return [
            {"name": "D1 Ada", "tokens": ["Ada", "works", "at", "OpenAI"], "tags": ["B-PER", "O", "O", "B-ORG"]},
            {"name": "D2 clean entities", "examples": [(["Ada", "joined", "OpenAI"], ["B-PER", "O", "B-ORG"]), (["Lin", "visits", "Paris"], ["B-PER", "O", "B-LOC"]), (["DeepMind", "hired", "Ilya"], ["B-ORG", "O", "B-PER"]), (["Mira", "met", "Sam"], ["B-PER", "O", "B-PER"]), (["Google", "opened", "office"], ["B-ORG", "O", "O"])]},
            {"name": "D3 adjacent padding", "examples": [(["Ada", "Bob", "spoke"], ["B-PER", "B-PER", "O"]), (["New", "York", "Labs"], ["B-LOC", "I-LOC", "B-ORG"]), (["Eve", "from", "IBM", "arrived"], ["B-PER", "O", "B-ORG", "O"]), (["Paris", "Paris", "team"], ["B-LOC", "B-ORG", "O"]), (["Zed", "Corp", "LLC"], ["B-ORG", "I-ORG", "I-ORG"])]},
            {"name": "D4 snippets", "examples": [(["Dr", "Chen", "at", "Mayo"], ["B-PER", "I-PER", "O", "B-ORG"]), (["Acme", "bought", "Beta"], ["B-ORG", "O", "B-ORG"]), (["Nina", "left", "Berlin"], ["B-PER", "O", "B-LOC"]), (["OpenAI", "San", "Francisco"], ["B-ORG", "B-LOC", "I-LOC"]), (["Raj", "from", "UN"], ["B-PER", "O", "B-ORG"])]},
            {"name": "D5 long multi", "examples": [(["Ada", "Lovelace", "worked", "with", "Babbage", "Labs"], ["B-PER", "I-PER", "O", "O", "B-ORG", "I-ORG"]), (["Maya", "from", "OpenAI", "visited", "London"], ["B-PER", "O", "B-ORG", "O", "B-LOC"]), (["IBM", "and", "Google", "met", "in", "Paris"], ["B-ORG", "O", "B-ORG", "O", "O", "B-LOC"]), (["Dr", "Lee", "joined", "Mayo", "Clinic"], ["B-PER", "I-PER", "O", "B-ORG", "I-ORG"]), (["Sam", "Altman", "visited", "New", "York"], ["B-PER", "I-PER", "O", "B-LOC", "I-LOC"])]},
        ]
    return [
        {"name": "D1 morphology", "tokens": ["walk", "walked", "walking"], "tags": ["VERB", "VERB", "VERB"]},
        {"name": "D2 clean POS", "examples": [(["dogs", "walk"], ["NOUN", "VERB"]), (["they", "walked"], ["NOUN", "VERB"]), (["walking", "helps"], ["NOUN", "VERB"]), (["red", "dogs", "run"], ["ADJ", "NOUN", "VERB"]), (["cats", "sleep"], ["NOUN", "VERB"])]},
        {"name": "D3 ambiguous", "examples": [(["walk", "home"], ["VERB", "NOUN"]), (["the", "walk"], ["DET", "NOUN"]), (["walking", "tour"], ["ADJ", "NOUN"]), (["we", "duck"], ["NOUN", "VERB"]), (["duck", "soup"], ["NOUN", "NOUN"])]},
        {"name": "D4 tagged lines", "examples": [(["bright", "birds", "sing"], ["ADJ", "NOUN", "VERB"]), (["users", "liked", "ads"], ["NOUN", "VERB", "NOUN"]), (["fast", "models", "serve"], ["ADJ", "NOUN", "VERB"]), (["the", "running", "joke"], ["DET", "ADJ", "NOUN"]), (["creators", "earned", "money"], ["NOUN", "VERB", "NOUN"])]},
        {"name": "D5 agreement", "examples": [(["the", "dogs", "were", "walking"], ["DET", "NOUN", "VERB", "VERB"]), (["a", "walk", "was", "short"], ["DET", "NOUN", "VERB", "ADJ"]), (["walking", "brands", "sell"], ["ADJ", "NOUN", "VERB"]), (["names", "ending", "ing", "confuse"], ["NOUN", "VERB", "NOUN", "VERB"]), (["past", "users", "walked"], ["ADJ", "NOUN", "VERB"])]},
    ]



POS_TAGS = ["NOUN", "VERB", "ADJ", "DET"]
POS_INDEX = {tag: i for i, tag in enumerate(POS_TAGS)}


def suffix_feature(word):
    low = word.lower()
    if low.endswith("ed"):
        return 1
    if low.endswith("ing"):
        return 2
    return 0


def pos_emission(token):
    scores = np.zeros(len(POS_TAGS))
    scores[POS_INDEX["NOUN"]] = 0.4
    low = token.lower()
    if low in {"the", "a"}:
        scores[POS_INDEX["DET"]] = 3.0
    if low in {"red", "bright", "fast", "short", "past", "walking", "running"}:
        scores[POS_INDEX["ADJ"]] = 1.6
    if low in {"walk", "walked", "walking", "run", "sleep", "liked", "serve", "earned", "were", "was", "sell", "confuse", "helps", "duck", "ending"}:
        scores[POS_INDEX["VERB"]] = 1.8
    if low.endswith("s") or low in {"home", "tour", "soup", "joke", "money", "walk", "duck", "ing"}:
        scores[POS_INDEX["NOUN"]] = 1.7
    return scores


def pos_transition_matrix():
    A = np.zeros((len(POS_TAGS), len(POS_TAGS)))
    A[POS_INDEX["DET"], POS_INDEX["NOUN"]] = 1.5
    A[POS_INDEX["DET"], POS_INDEX["ADJ"]] = 1.0
    A[POS_INDEX["ADJ"], POS_INDEX["NOUN"]] = 1.2
    A[POS_INDEX["NOUN"], POS_INDEX["VERB"]] = 1.4
    A[POS_INDEX["VERB"], POS_INDEX["ADJ"]] = 0.4
    return A


def pos_tagger(tokens, suffix_features=None, A=None):
    suffix_features = suffix_features if suffix_features is not None else [suffix_feature(tok) for tok in tokens]
    A = pos_transition_matrix() if A is None else A
    emissions = []
    for token, feat in zip(tokens, suffix_features):
        score = pos_emission(token)
        if feat == 1:
            score[POS_INDEX["VERB"]] += 0.8
        if feat == 2:
            score[POS_INDEX["VERB"]] += 0.4
            score[POS_INDEX["ADJ"]] += 0.4
        emissions.append(score)
    emissions = np.vstack(emissions)
    dp = emissions[0].copy()
    back = []
    for t in range(1, len(tokens)):
        scores = dp[:, None] + A + emissions[t][None, :]
        back.append(np.argmax(scores, axis=0))
        dp = np.max(scores, axis=0)
    idx = int(np.argmax(dp))
    path = [idx]
    for bp in reversed(back):
        idx = int(bp[idx])
        path.append(idx)
    return [POS_TAGS[i] for i in reversed(path)], emissions


def token_accuracy(pred, gold):
    return accuracy(pred, gold)


def partial_feature_report(pred_bundles, gold_bundles):
    hits = 0
    total = 0
    for pred, gold in zip(pred_bundles, gold_bundles):
        for key, value in gold.items():
            total += 1
            hits += int(pred.get(key) == value)
    return hits / max(1, total)


## The concept, built once: emissions plus transitions

The tagger uses $p(y_t=k|x_t)=\exp(e_t(k))/\sum_j\exp(e_t(j))$ and $s(y)=\sum_t e_t(y_t)+\sum_{t>1}A_{y_{t-1},y_t}$. The lesson suffix features are exact.

In [ ]:

words = ["walk", "walked", "walking"]
features = [suffix_feature(word) for word in words]
print(features)
assert features == [0, 1, 2]


Local scores can be overturned by context. The lesson's local $[0.55,0.45]$ can become contextual $[0.234043,0.765957]$, so morphology is evidence rather than destiny.

In [ ]:

probs = softmax([0.2, 2.0, 0.1])
contextual = softmax([0.0, 1.1856211732351492])
local = [0.55, 0.45]
print(np.round(probs, 6))
print(np.round(contextual, 6))
assert np.allclose(np.round(probs, 6), [0.125715, 0.760533, 0.113752])
assert local == [0.55, 0.45]
assert np.round(contextual, 6).tolist() == [0.234043, 0.765957]


## The dataset ladder

D1 is the morphology toy, then clean POS sentences, ambiguous words and suffix traps, inline tagged sentences, and D5 agreement-sensitive examples.

In [ ]:

ladder = make_sequence_ladder("8.18")
for rung in ladder:
    examples = rung.get("examples", [(rung["tokens"], rung["tags"])])
    print(rung["name"], "examples=", len(examples), "sample=", examples[0])


In [ ]:

rows = []
metrics = []
ambiguity = []
for rung in ladder:
    examples = rung.get("examples", [(rung["tokens"], rung["tags"])])
    scores = []
    ambiguous = 0
    for tokens, gold in examples:
        pred, emissions = pos_tagger(tokens)
        scores.append(token_accuracy(pred, gold))
        ambiguous += sum(int(max(softmax(row)) < 0.7) for row in emissions)
    score = sum(scores) / len(scores)
    metrics.append(score)
    ambiguity.append(ambiguous)
    rows.append([rung["name"], len(examples), round(score, 3), ambiguous])
show_table(rows, ["rung", "examples", "accuracy", "ambiguous"])


In [ ]:

fig, axes = plt.subplots(1, 6, figsize=(15, 2.4))
for ax, rung in zip(axes[:5], ladder):
    tokens, gold = rung.get("examples", [(rung["tokens"], rung["tags"])])[0]
    pred, emissions = pos_tagger(tokens)
    ax.imshow(emissions, cmap="cividis", aspect="auto")
    ax.set_title(rung["name"].split()[0])
    ax.set_xlabel("POS")
    ax.set_ylabel("token")
axes[5].plot(ambiguity, metrics, marker="o")
axes[5].set_ylim(0, 1.05)
axes[5].set_title("accuracy vs ambiguity")
axes[5].set_xlabel("ambiguous tokens")
fig.tight_layout()


## Pitfall on D5: overtrusting suffixes

The suffix value 2 for walking is useful, but domain names and adjectives can share that ending. Exact-bundle-only evaluation also hides partial morphology correctness.

In [ ]:

d5 = ladder[-1]
tokens, gold = d5["examples"][2]
wrong_A = np.zeros((len(POS_TAGS), len(POS_TAGS)))
wrong_pred, wrong_emissions = pos_tagger(tokens, A=wrong_A)
right_pred, right_emissions = pos_tagger(tokens)
pred_bundles = [{"POS": right_pred[0], "Tense": "Past"}, {"POS": right_pred[1], "Number": "Plur"}]
gold_bundles = [{"POS": "ADJ", "Tense": "Past"}, {"POS": "NOUN", "Number": "Plur"}]
partial = partial_feature_report(pred_bundles, gold_bundles)
print("tokens", tokens)
print("wrong", wrong_pred, token_accuracy(wrong_pred, gold))
print("right", right_pred, token_accuracy(right_pred, gold))
print("partial bundle", partial)
assert suffix_feature("walking") == 2
assert token_accuracy(right_pred, gold) >= token_accuracy(wrong_pred, gold)
assert 0 < partial <= 1


## Evaluate it + Practice

- Metric: token accuracy; no-skill baseline predicts the most common POS.
- Sanity check: walk/walked/walking must map to suffix features 0/1/2.
- Ablation: remove transitions and ambiguous dictionary words should degrade.
- Failure signal: exact bundle accuracy hides partially correct POS or tense.

Practice: add Number=Sing/Plur bundles to D5 and report partial credit.

Practice: make a list of names ending in ing and test suffix overtrust.